## Exploring

In [8]:
import xarray as xr

file = "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/merra2/level2_4/MERRA2_400.tavg1_2d_lnd_Nx.20191115.SUB.nc"

data = xr.open_dataset(file)

data

<xarray.Dataset> Size: 60MB
Dimensions:  (time: 24, lon: 576, lat: 361)
Coordinates:
  * time     (time) datetime64[ns] 192B 2019-11-15T00:30:00 ... 2019-11-15T23...
  * lon      (lon) float64 5kB -180.0 -179.4 -178.8 -178.1 ... 178.1 178.8 179.4
  * lat      (lat) float64 3kB -90.0 -89.5 -89.0 -88.5 ... 88.5 89.0 89.5 90.0
Data variables:
    TSOIL2   (time, lat, lon) float32 20MB ...
    TSOIL3   (time, lat, lon) float32 20MB ...
    TSOIL4   (time, lat, lon) float32 20MB ...
Attributes: (12/32)
    CDI:                               Climate Data Interface version 1.9.8 (...
    Conventions:                       CF-1
    History:                           Original file generated: Mon Dec 16 07...
    Comment:                           GMAO filename: d5124_m2_jan10.tavg1_2d...
    Filename:                          MERRA2_400.tavg1_2d_lnd_Nx.20191115.nc4
    Institution:                       NASA Global Modeling and Assimilation ...
    ...                                ...
    RangeBeginningDate:                2019-11-15
    RangeBeginningTime:                00:00:00.000000
    RangeEndingDate:                   2019-11-15
    RangeEndingTime:                   23:59:59.000000
    history_L34RS:                     'Created by L34RS v1.4.4 @ NASA GES DI...
    CDO:                               Climate Data Operators version 1.9.8 (...

## checking the ts values in all 3 layers for a specific coordinates

In [9]:
import xarray as xr

# Your dataset
data = xr.open_dataset(file)

# Method 1: Select by lat/lon coordinates (easiest)
lat_target = 48.9  # Paris latitude
lon_target = 2.3  # Paris longitude

# Get values for both layers at that location
tsoil2 = data['TSOIL2'].sel(lat=lat_target, lon=lon_target, method='nearest')
tsoil3 = data['TSOIL3'].sel(lat=lat_target, lon=lon_target, method='nearest')
tsoil4 = data['TSOIL4'].sel(lat=lat_target, lon=lon_target, method='nearest')

print(f"Location: {lat_target}°N, {lon_target}°E")
print(f"9.9-29cm layer: {tsoil2.values[0]:.2f} K")
print(f"29-68cm layer: {tsoil3.values[0]:.2f} K")
print(f"68-144cm layer: {tsoil4.values[0]:.2f} K")

Location: 48.9°N, 2.3°E
9.9-29cm layer: 279.75 K
29-68cm layer: 282.13 K
68-144cm layer: 285.25 K


## For whole day for upper specific location

- And compared with the combined file and all are correctely computed

- The half hourly, root zone and values after break are all properly computed

In [10]:
for i, time_val in enumerate(data.time):
    temp_data = data.isel(time=i).sel(lat=37.2134, lon=-98.0969, method='nearest')
    print(f"Hour {i:2d} ({time_val.values}): "
          f"TSOIL2={temp_data['TSOIL2'].values:.2f}K, "
          f"TSOIL3={temp_data['TSOIL3'].values:.2f}K, "
          f"TSOIL4={temp_data['TSOIL4'].values:.2f}K")

Hour  0 (2019-11-15T00:30:00.000000000): TSOIL2=277.66K, TSOIL3=281.13K, TSOIL4=286.62K
Hour  1 (2019-11-15T01:30:00.000000000): TSOIL2=277.76K, TSOIL3=281.12K, TSOIL4=286.61K
Hour  2 (2019-11-15T02:30:00.000000000): TSOIL2=277.82K, TSOIL3=281.11K, TSOIL4=286.60K
Hour  3 (2019-11-15T03:30:00.000000000): TSOIL2=277.84K, TSOIL3=281.11K, TSOIL4=286.59K
Hour  4 (2019-11-15T04:30:00.000000000): TSOIL2=277.82K, TSOIL3=281.10K, TSOIL4=286.58K
Hour  5 (2019-11-15T05:30:00.000000000): TSOIL2=277.77K, TSOIL3=281.09K, TSOIL4=286.57K
Hour  6 (2019-11-15T06:30:00.000000000): TSOIL2=277.71K, TSOIL3=281.09K, TSOIL4=286.56K
Hour  7 (2019-11-15T07:30:00.000000000): TSOIL2=277.63K, TSOIL3=281.08K, TSOIL4=286.55K
Hour  8 (2019-11-15T08:30:00.000000000): TSOIL2=277.54K, TSOIL3=281.07K, TSOIL4=286.54K
Hour  9 (2019-11-15T09:30:00.000000000): TSOIL2=277.44K, TSOIL3=281.06K, TSOIL4=286.53K
Hour 10 (2019-11-15T10:30:00.000000000): TSOIL2=277.33K, TSOIL3=281.05K, TSOIL4=286.52K
Hour 11 (2019-11-15T11:30:00.000

## Test: Combining MERRA2 with insitu

- Test script works fine, and is generlized on all networks on cluster

In [5]:
#!/usr/bin/env python3
"""
MERRA2 + In-situ Soil Temperature Combination Script
Modified from GLDAS script to handle MERRA2 3-layer soil temperature data

MERRA2 Layers:
- TSOIL2: 9.9-29.4cm → ts_m2_l1
- TSOIL3: 29.4-68.0cm → ts_m2_l2  
- TSOIL4: 68.0-144.3cm → ts_m2_l3

Root zone (10-100cm) weights:
- Layer 2: 0.194m (21.6%)
- Layer 3: 0.386m (42.9%)
- Layer 4: 0.320m (35.6%) [truncated at 100cm]
"""

import os
import pandas as pd
import xarray as xr
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')


def find_nearest_merra2_grid(target_lat, target_lon, merra2_lats, merra2_lons):
    """Find nearest MERRA2 grid point."""
    lat_idx = int(np.argmin(np.abs(merra2_lats - target_lat)))
    lon_idx = int(np.argmin(np.abs(merra2_lons - target_lon)))
    actual_lat = float(merra2_lats[lat_idx])
    actual_lon = float(merra2_lons[lon_idx])
    return lat_idx, lon_idx, actual_lat, actual_lon


def extract_merra2_for_station(merra2_dir, target_lat, target_lon, start_date, end_date):
    """
    Extract MERRA2 half-hourly data for station and convert to hourly using averaging.
    Returns dict{datetime: {ts_m2_l1, ts_m2_l2, ts_m2_l3}} and grid coordinates.
    """
    print(f"  Extracting MERRA2 data for lat={target_lat:.4f}, lon={target_lon:.4f}")
    print(f"  Date range: {start_date} to {end_date}")
    
    # Convert dates
    start_date = pd.to_datetime(start_date).normalize()
    end_date = pd.to_datetime(end_date).normalize()
    
    merra2_data = {}  # {datetime: {ts_m2_l1, ts_m2_l2, ts_m2_l3}}
    grid_coords = None
    
    # Process each day
    current_date = start_date
    while current_date <= end_date:
        date_str = current_date.strftime("%Y%m%d")
        file_path = os.path.join(merra2_dir, f"MERRA2_400.tavg1_2d_lnd_Nx.{date_str}.SUB.nc")
        
        if not os.path.exists(file_path):
            print(f"    Missing MERRA2 file: {file_path}")
            current_date += timedelta(days=1)
            continue
            
        print(f"    Reading: MERRA2_400.tavg1_2d_lnd_Nx.{date_str}.SUB.nc")
        
        try:
            with xr.open_dataset(file_path) as ds:
                lats = ds["lat"].values
                lons = ds["lon"].values
                
                # Find grid point (only once)
                if grid_coords is None:
                    lat_idx, lon_idx, actual_lat, actual_lon = find_nearest_merra2_grid(
                        target_lat, target_lon, lats, lons
                    )
                    grid_coords = (actual_lat, actual_lon)
                    print(f"    MERRA2 grid: lat={actual_lat:.2f}, lon={actual_lon:.2f}")
                else:
                    # Recompute indices for this day
                    lat_idx = int(np.argmin(np.abs(lats - grid_coords[0])))
                    lon_idx = int(np.argmin(np.abs(lons - grid_coords[1])))
                
                # Extract soil temperature layers at station location
                tsoil2 = ds["TSOIL2"].isel(lat=lat_idx, lon=lon_idx).values  # ts_m2_l1
                tsoil3 = ds["TSOIL3"].isel(lat=lat_idx, lon=lon_idx).values  # ts_m2_l2
                tsoil4 = ds["TSOIL4"].isel(lat=lat_idx, lon=lon_idx).values  # ts_m2_l3
                times = pd.to_datetime(ds["time"].values)
                
                # Convert half-hourly to hourly using averaging strategy
                half_hourly_data = {}
                for i, (t, t2, t3, t4) in enumerate(zip(times, tsoil2, tsoil3, tsoil4)):
                    t = pd.Timestamp(t).floor("S")
                    
                    # Quality check: handle NaN, -9999, and unreasonable temperatures
                    def is_valid_temp(temp):
                        if pd.isna(temp) or temp == -9999 or temp <= 0:
                            return False
                        return 200 <= temp <= 350  # Reasonable temperature range in Kelvin
                    
                    # Only store if ALL three layers have valid data
                    if is_valid_temp(t2) and is_valid_temp(t3) and is_valid_temp(t4):
                        half_hourly_data[t] = {
                            'ts_m2_l1': float(t2),
                            'ts_m2_l2': float(t3), 
                            'ts_m2_l3': float(t4)
                        }
                
                # Generate hourly timestamps for this day
                day_start = current_date
                for hour in range(24):
                    hourly_dt = day_start + timedelta(hours=hour)
                    
                    # Apply half-hourly averaging strategy
                    prev_half = hourly_dt - timedelta(minutes=30)  # HH-1:30
                    next_half = hourly_dt + timedelta(minutes=30)  # HH:30
                    
                    prev_data = half_hourly_data.get(prev_half)
                    next_data = half_hourly_data.get(next_half)
                    
                    if prev_data and next_data:
                        # Average both half-hourly values
                        merra2_data[hourly_dt] = {
                            'ts_m2_l1': (prev_data['ts_m2_l1'] + next_data['ts_m2_l1']) / 2.0,
                            'ts_m2_l2': (prev_data['ts_m2_l2'] + next_data['ts_m2_l2']) / 2.0,
                            'ts_m2_l3': (prev_data['ts_m2_l3'] + next_data['ts_m2_l3']) / 2.0
                        }
                    elif prev_data:
                        # Use only previous half-hour
                        merra2_data[hourly_dt] = prev_data.copy()
                    elif next_data:
                        # Use only next half-hour  
                        merra2_data[hourly_dt] = next_data.copy()
                    # If neither exists (missing/invalid data), skip this hour completely
                    # This will result in NaN values in the final merge
                        
        except Exception as e:
            print(f"    Error reading {file_path}: {e}")
            
        current_date += timedelta(days=1)
    
    print(f"  Collected {len(merra2_data)} hourly MERRA2 records")
    return merra2_data, grid_coords


def compute_merra2_root_zone(ts_m2_l1, ts_m2_l2, ts_m2_l3):
    """
    Compute root zone soil temperature (10-100cm) from MERRA2 3 layers.
    
    Weights based on depth coverage:
    - Layer 2 (9.9-29.4cm): usable 10-29.4cm = 19.4cm → 21.6%
    - Layer 3 (29.4-68cm): full layer = 38.6cm → 42.9%  
    - Layer 4 (68-144cm): usable 68-100cm = 32.0cm → 35.6%
    """
    if pd.isna(ts_m2_l1) or pd.isna(ts_m2_l2) or pd.isna(ts_m2_l3):
        return np.nan
        
    # Depth weights for root zone (10-100cm)
    d1 = 0.194  # Layer 2: 10-29.4cm usable
    d2 = 0.386  # Layer 3: 29.4-68cm full
    d3 = 0.320  # Layer 4: 68-100cm usable (truncated)
    total = 0.90  # Total root zone depth
    
    ts_merra2 = (ts_m2_l1 * d1 + ts_m2_l2 * d2 + ts_m2_l3 * d3) / total
    return round(ts_merra2, 3)


def process_station_merra2_insitu(station_csv_path, merra2_dir, output_dir):
    """Process single station: combine in-situ data with MERRA2."""
    print(f"\n🌡️ Processing station: {station_csv_path}")
    
    # Read in-situ data
    try:
        df = pd.read_csv(station_csv_path)
        print(f"  In-situ records: {len(df)}")
    except Exception as e:
        print(f"  ❌ Error reading CSV: {e}")
        return False
    
    # Check required columns
    required_cols = ['lat', 'lon', 'datetime']
    if 'datetime' not in df.columns and 'date' in df.columns and 'time' in df.columns:
        # Create datetime from date + time
        df['datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['time'].astype(str))
    
    if not all(col in df.columns for col in required_cols):
        print(f"  ❌ Missing required columns: {required_cols}")
        return False
    
    # Get station coordinates
    station_lat = df['lat'].iloc[0]
    station_lon = df['lon'].iloc[0]
    
    # Determine date range
    df['datetime'] = pd.to_datetime(df['datetime'])
    start_date = df['datetime'].min().date()
    end_date = df['datetime'].max().date()
    
    print(f"  Station: lat={station_lat:.4f}, lon={station_lon:.4f}")
    print(f"  Date range: {start_date} to {end_date}")
    
    # Extract MERRA2 data
    merra2_data, grid_coords = extract_merra2_for_station(
        merra2_dir, station_lat, station_lon, start_date, end_date
    )
    
    if not merra2_data:
        print(f"  ❌ No MERRA2 data found")
        return False
    
    # Create MERRA2 DataFrame
    merra2_records = []
    for dt, temps in merra2_data.items():
        # Compute root zone
        ts_merra2 = compute_merra2_root_zone(
            temps['ts_m2_l1'], temps['ts_m2_l2'], temps['ts_m2_l3']
        )
        
        merra2_records.append({
            'datetime': dt,
            'ts_m2_l1': round(temps['ts_m2_l1'], 3),
            'ts_m2_l2': round(temps['ts_m2_l2'], 3),
            'ts_m2_l3': round(temps['ts_m2_l3'], 3),
            'ts_merra2': ts_merra2,
            'lat_merra2': grid_coords[0],
            'lon_merra2': grid_coords[1]
        })
    
    merra2_df = pd.DataFrame(merra2_records)
    merra2_df['datetime'] = pd.to_datetime(merra2_df['datetime'])
    
    print(f"  MERRA2 records: {len(merra2_df)}")
    
    # Merge with in-situ data
    df['datetime'] = pd.to_datetime(df['datetime'])
    merged_df = df.merge(merra2_df, on='datetime', how='left')
    
    print(f"  Merged records: {len(merged_df)}")
    print(f"  MERRA2 matches: {merged_df['ts_merra2'].notna().sum()}")
    
    # Drop specified columns
    # 1. Drop ts_station column
    if 'ts_station' in merged_df.columns:
        merged_df = merged_df.drop(columns=['ts_station'])
    
    # 2. Drop all T_depth columns (e.g., T_0.200_0.200, T_0.500_0.500, etc.)
    t_columns = [col for col in merged_df.columns if col.startswith('T_')]
    if t_columns:
        merged_df = merged_df.drop(columns=t_columns)
        print(f"  Dropped T_ columns: {t_columns}")
    
    # 3. Reorder columns - place ts_merra2 after ts_station_k
    columns = merged_df.columns.tolist()
    if 'ts_merra2' in columns and 'ts_station_k' in columns:
        # Remove ts_merra2 from its current position
        columns.remove('ts_merra2')
        # Find position of ts_station_k and insert ts_merra2 after it
        station_k_idx = columns.index('ts_station_k')
        columns.insert(station_k_idx + 1, 'ts_merra2')
        # Reorder DataFrame
        merged_df = merged_df[columns]
    
    # 4. Paired NaN logic: if MERRA2 is missing/invalid, set station data to NaN too
    # This ensures both datasets have data at the same timestamps
    if 'ts_station_k' in merged_df.columns:
        # Where MERRA2 data is missing, set station data to NaN
        merra2_missing = merged_df['ts_merra2'].isna()
        if merra2_missing.any():
            merged_df.loc[merra2_missing, 'ts_station_k'] = pd.NA
            print(f"  Set {merra2_missing.sum()} station values to NaN where MERRA2 missing")
        
        # Where station data is missing, set MERRA2 to NaN (reciprocal)
        station_missing = merged_df['ts_station_k'].isna()
        if station_missing.any():
            merged_df.loc[station_missing, ['ts_merra2', 'ts_m2_l1', 'ts_m2_l2', 'ts_m2_l3']] = pd.NA
            print(f"  Set {station_missing.sum()} MERRA2 values to NaN where station missing")
    
    # Create output directory and save
    os.makedirs(output_dir, exist_ok=True)
    
    # Keep same filename (no network name addition)
    input_filename = os.path.basename(station_csv_path)
    output_filename = input_filename.replace('.csv', '_merra2.csv')
    output_path = os.path.join(output_dir, output_filename)
    
    # Save merged data
    merged_df.to_csv(output_path, index=False)
    print(f"  ✅ Saved: {output_path}")
    
    # Print sample results
    sample = merged_df[['datetime', 'ts_merra2', 'ts_m2_l1', 'ts_m2_l2', 'ts_m2_l3']].dropna().head(3)
    if not sample.empty:
        print(f"  📊 Sample results:")
        for _, row in sample.iterrows():
            print(f"    {row['datetime']}: ts_merra2={row['ts_merra2']:.3f}K "
                  f"(L1={row['ts_m2_l1']:.3f}, L2={row['ts_m2_l2']:.3f}, L3={row['ts_m2_l3']:.3f})")
    
    return True


def main():
    """Test single station processing."""
    # Paths
    station_csv = "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/in_situ/combine/ARM/Anthony/Anthony_soil_temperature_depths.csv"
    merra2_dir = "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/merra2/level2_4"
    output_dir = "/home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/merra2/combine/ARM/Anthony"
    
    print("🚀 MERRA2 + In-situ Soil Temperature Combination")
    print("=" * 60)
    
    success = process_station_merra2_insitu(station_csv, merra2_dir, output_dir)
    
    if success:
        print("\n✅ Processing completed successfully!")
    else:
        print("\n❌ Processing failed!")


if __name__ == "__main__":
    main()

🚀 MERRA2 + In-situ Soil Temperature Combination

🌡️ Processing station: /home/aminr/Desktop/Master_Thesis/Global_scale/bottom_layer/in_situ/combine/ARM/Anthony/Anthony_soil_temperature_depths.csv
  In-situ records: 42223
  Station: lat=37.2134, lon=-98.0969
  Date range: 2016-03-07 to 2020-12-31
  Extracting MERRA2 data for lat=37.2134, lon=-98.0969
  Date range: 2016-03-07 to 2020-12-31
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160307.SUB.nc
    MERRA2 grid: lat=37.00, lon=-98.12
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160308.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160309.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160310.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160311.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160312.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160313.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160314.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160315.SUB.nc
    Reading: MERRA2_400.tavg1_2d_lnd_Nx.20160316.SUB.nc
    Readin

## Checking the combined files that are transfered from cluster

In [1]:
import os
import glob
from pathlib import Path
import pandas as pd

ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/combine")

REQUIRED_COLS = {
    "date","time","ts_station_k","ts_merra2",
    "lat","lon","elev","cc","lc",
    "land_cover_group","climate_group","temp_class","elevation_class",
    "datetime","ts_m2_l1","ts_m2_l2","ts_m2_l3",
    "lat_merra2","lon_merra2",
}

records = []
problems = []

for net_dir in sorted(ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name

    for st_dir in sorted(net_dir.iterdir()):
        if not st_dir.is_dir():
            continue
        station = st_dir.name

        for f in glob.glob(str(st_dir / "*.csv")):
            rec = {"network": network, "station": station, "file": f}
            try:
                df = pd.read_csv(f)
            except Exception as e:
                rec["status"] = "unreadable"
                rec["issue"] = f"read_error: {e}"
                records.append(rec)
                problems.append(rec)
                continue

            rec["status"] = "ok"
            missing = [c for c in REQUIRED_COLS if c not in df.columns]
            if missing:
                rec["status"] = "missing_columns"
                rec["issue"] = "missing: " + ",".join(missing)
                problems.append(rec)

            if df.empty:
                rec["status"] = "empty_file"
                rec["issue"] = "no rows"
                problems.append(rec)
            else:
                n_valid_station = df["ts_station_k"].notna().sum() if "ts_station_k" in df else 0
                n_valid_merra2  = df["ts_merra2"].notna().sum()   if "ts_merra2"  in df else 0
                if n_valid_station == 0 and n_valid_merra2 == 0:
                    rec["status"] = "no_valid_ts"
                    rec["issue"] = "ts_station_k & ts_merra2 all NaN"
                    problems.append(rec)

            records.append(rec)

df_all = pd.DataFrame(records)

n_networks = df_all["network"].nunique()
n_stations = df_all[["network","station"]].drop_duplicates().shape[0]
n_files    = len(df_all)

print(f"Networks: {n_networks}")
print(f"Stations: {n_stations}")
print(f"Files:    {n_files}")

print("\nProblematic files by status:")
print(df_all["status"].value_counts())

# write reports
df_all.to_csv(ROOT / "audit_merra2_combine_listing.csv", index=False)
if problems:
    pd.DataFrame(problems).to_csv(ROOT / "audit_merra2_combine_problems.csv", index=False)
    print("\nSaved detailed problems to audit_merra2_combine_problems.csv")
else:
    print("\nAll files have required columns and at least one valid ts_station_k or ts_merra2.")


Networks: 34
Stations: 1173
Files:    1173

Problematic files by status:
status
ok    1173
Name: count, dtype: int64

All files have required columns and at least one valid ts_station_k or ts_merra2.


## Computing the SD and correlation for merra2 combine files

In [2]:
import pandas as pd
from pathlib import Path

# Root dirs
M2_COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/combine")
OUT_SUMMARY     = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/merra2_insitu_sd_corr_summary.csv")

results = []

for network_dir in sorted(M2_COMBINE_ROOT.iterdir()):
    if not network_dir.is_dir():
        continue
    network = network_dir.name

    for station_dir in sorted(network_dir.iterdir()):
        if not station_dir.is_dir():
            continue
        station = station_dir.name

        csv_files = list(station_dir.glob("*.csv"))
        if not csv_files:
            print(f"[WARN] No CSV in {station_dir}, skipping.")
            continue

        csv_path = csv_files[0]

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"[ERROR] Cannot read {csv_path}: {e}")
            results.append({
                "network": network,
                "station": station,
                "n_points": 0,
                "sd_insitu": None,
                "sd_merra2": None,
                "sd_ratio_merra2_insitu": None,
                "corr_merra2_insitu": None,
                "pass_sd_filter": False,
                "pass_corr_filter": False,
            })
            continue

        # make datetime if useful later (optional)
        if "date" in df.columns and "time" in df.columns and "datetime" not in df.columns:
            df["datetime"] = pd.to_datetime(
                df["date"].astype(str) + " " + df["time"].astype(str),
                errors="coerce"
            )

        # required columns
        if not {"ts_station_k", "ts_merra2"}.issubset(df.columns):
            print(f"[WARN] Missing ts_station_k or ts_merra2 in {csv_path}, skipping station.")
            continue

        # pairwise valid values only
        sub = df[["ts_station_k", "ts_merra2"]].dropna()
        n_points = len(sub)

        if n_points < 2:
            print(f"[WARN] Not enough valid points for {network}/{station} ({n_points}), skipping stats.")
            sd_insitu = sd_m2 = ratio = corr = None
            pass_sd = pass_corr = False
        else:
            sd_insitu = sub["ts_station_k"].std()
            sd_m2     = sub["ts_merra2"].std()
            ratio = sd_m2 / sd_insitu if sd_insitu and sd_insitu > 0 else None
            corr  = sub["ts_station_k"].corr(sub["ts_merra2"])

            # same filters as ERA5 case
            pass_sd   = (ratio is not None) and (0.1 <= ratio <= 3)
            pass_corr = (corr is not None) and (corr >= 0.5)

        results.append({
            "network": network,
            "station": station,
            "n_points": n_points,
            "sd_insitu": sd_insitu,
            "sd_merra2": sd_m2,
            "sd_ratio_merra2_insitu": ratio,
            "corr_merra2_insitu": corr,
            "pass_sd_filter": pass_sd,
            "pass_corr_filter": pass_corr,
        })

# Build DataFrame
summary_df = pd.DataFrame(results)

# Round numeric columns
num_cols = ["sd_insitu", "sd_merra2", "sd_ratio_merra2_insitu", "corr_merra2_insitu"]
for col in num_cols:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].round(3)

summary_df.to_csv(OUT_SUMMARY, index=False)
print(f"Saved summary to {OUT_SUMMARY}")


Saved summary to /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/merra2_insitu_sd_corr_summary.csv


In [3]:
import pandas as pd
from pathlib import Path

summary_path = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/merra2_insitu_sd_corr_summary.csv")

df = pd.read_csv(summary_path)

# Total stations
total = len(df)

# Stations that pass both SD and correlation filters
pass_both = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]

# Stations failing at least one filter
fail_any = df[(df["pass_sd_filter"] == False) | (df["pass_corr_filter"] == False)]

print(f"Total stations          : {total}")
print(f"Pass both (SD & corr)   : {len(pass_both)}")
print(f"Fail SD or corr (or both): {len(fail_any)}")

# Optional: counts per network
print("\nPer network (pass both):")
print(pass_both.groupby("network")["station"].count())

print("\nPer network (fail any):")
print(fail_any.groupby("network")["station"].count())

# NEW: print failing station names per network
print("\nFailing stations by network:")
for network, sub in fail_any.groupby("network"):
    station_list = sorted(sub["station"].unique())
    print(f"{network}: {station_list} = {len(station_list)}")


Total stations          : 1173
Pass both (SD & corr)   : 1156
Fail SD or corr (or both): 17

Per network (pass both):
network
ARM                   14
BIEBRZA_S-1           18
BNZ-LTER               9
Berlin                 1
COSMOS-UK             49
CTP_SMTMN             57
CW3E                   8
DAHRA                  1
FLUXNET-AMERIFLUX      5
FMI                   10
FR_Aqui                5
HOAL                  29
HOBE                  30
LABFLUX                1
MAQU                  16
MOL-RAO                2
NAQU                  10
NGARI                 20
ORACLE                 1
RISMA                 23
ROMPS                  5
Ru_CFR                 2
SCAN                 203
SKKU                   1
SMN-SDR               33
SMOSMANIA             22
SNOTEL               438
STEMS                  2
TERENO                 5
TWENTE                25
USCRN                 92
WEGENERNET            11
XMS-CAT                8
Name: station, dtype: int64

Per network (fail an

## Taking pixel mean

In [1]:
import os
import glob
import pandas as pd
from collections import Counter
from pathlib import Path

# ---------------------------------------------------------------------
# Paths for MERRA-2 sub-surface
# ---------------------------------------------------------------------

COMBINE_ROOT = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/combine"
OUT_ROOT     = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean"
SUMMARY_PATH = "/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/merra2_insitu_sd_corr_summary.csv"

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def majority(series):
    """Most common non-NaN value in a series."""
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return pd.NA
    return Counter(vals).most_common(1)[0][0]

def classify_elevation(elev):
    """Classify elevation using DEM-I/II/III/IV thresholds."""
    if pd.isna(elev):
        return "DEM-Unknown"
    try:
        elevation = float(elev)
    except Exception:
        return "DEM-Unknown"

    if elevation < 635.523:
        return "DEM-I"
    elif elevation < 1237.128:
        return "DEM-II"
    elif elevation < 1838.733:
        return "DEM-III"
    else:
        return "DEM-IV"

def load_pass_stations(summary_path):
    """
    Read summary CSV and return a set of (network, station) pairs
    that passed BOTH SD and correlation filters.
    """
    df = pd.read_csv(summary_path)
    ok = df[(df["pass_sd_filter"] == True) & (df["pass_corr_filter"] == True)]
    return set(zip(ok["network"], ok["station"]))

# ---------------------------------------------------------------------
# Standard processing for moderate-size networks
# ---------------------------------------------------------------------

def process_network(network, pass_stations):
    base_in   = os.path.join(COMBINE_ROOT, network)
    base_out  = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out  = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)

    print(f"\n=== Network: {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    all_records = []
    used_station_dirs = []

    # Load station data, but only for stations that passed filters
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            print(f"  Skipping {network}/{station_name} (failed filter in summary).")
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            # Required columns for MERRA-2 sub-surface
            required_cols = {
                "date", "time",
                "ts_station_k", "ts_merra2",
                "lat", "lon", "elev",
                "cc", "lc",
                "land_cover_group", "climate_group", "temp_class",
                "elevation_class",
                "lat_merra2", "lon_merra2",
            }
            if not required_cols.issubset(df.columns):
                print(f"  [WARN] Missing required columns in {f}, skipping this file.")
                continue

            df["station"] = station_name
            all_records.append(df)
            used_station_dirs.append(station_dir)

    if not all_records:
        print(f"No valid CSV files found under {base_in}/* (after filtering), skipping.")
        return

    df_all = pd.concat(all_records, ignore_index=True)
    print(f"Loaded {len(set(used_station_dirs))} stations after filtering, total rows: {len(df_all)}")

    # DENSE PIXELS: at least 2 stations in same (lat_merra2, lon_merra2)
    pixel_station_counts = (
        df_all.groupby(["lat_merra2", "lon_merra2"])["station"]
              .nunique()
              .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_merra2", "lon_merra2"]]
    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    n_sparse_files = 0
    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # DENSE
    print("Processing dense pixels (pixel-mean files)...")
    df_dense_all = df_all.merge(dense_pixels, on=["lat_merra2", "lon_merra2"], how="inner")
    print("Rows in dense pixels (before grouping):", len(df_dense_all))

    stations_in_dense = set()

    # Also read existing dense files to populate stations_in_dense (idempotent)
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    if not df_dense_all.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_merra2"]
            plon = pix["lon_merra2"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            # Skip this pixel if its dense file already exists
            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_merra2={plat}, lon_merra2={plon} (file exists).")
                continue

            df_pix = df_dense_all[
                (df_dense_all["lat_merra2"] == plat) &
                (df_dense_all["lon_merra2"] == plon)
            ].copy()

            if df_pix.empty:
                continue

            print(f"  Dense pixel at lat_merra2={plat}, lon_merra2={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    ts_ref = g_valid["ts_station_k"].mean()
                    ts_m2  = g_valid["ts_merra2"].iloc[0]

                    lat_val  = majority(g_valid["lat"])
                    lon_val  = majority(g_valid["lon"])
                    cc_val   = majority(g_valid["cc"])
                    lc_val   = majority(g_valid["lc"])
                    lcg_val  = majority(g_valid["land_cover_group"])
                    clim_val = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref = pd.NA
                    ts_m2  = pd.NA
                    lat_val = pd.NA
                    lon_val = pd.NA
                    cc_val = pd.NA
                    lc_val = pd.NA
                    lcg_val = pd.NA
                    clim_val = pd.NA
                    tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,
                    "ts_merra2": ts_m2,
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_merra2"]    = pd.to_numeric(df_pix_mean["ts_merra2"],    errors="coerce").round(3)

            df_pix_mean["lat_merra2"] = plat
            df_pix_mean["lon_merra2"] = plon
            df_pix_mean["stations"]   = stations_str

            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")
    else:
        print("No dense pixels found to process.")

    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")

    # SPARSE: station-wise files, excluding stations used in any dense pixel
    print("Writing sparse (station-wise) files (excluding dense stations)...")
    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue

        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            df = df.rename(columns=rename_map)

            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Stations found (raw): {n_stations_found}")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")

# ---------------------------------------------------------------------
# SNOTEL-specific processing (memory-safe)
# ---------------------------------------------------------------------

def process_network_snotel(pass_stations):
    network = "SNOTEL"
    base_in   = os.path.join(COMBINE_ROOT, network)
    base_out  = os.path.join(OUT_ROOT, network)
    sparse_out = os.path.join(base_out, "sparse")
    dense_out  = os.path.join(base_out, "dense")
    os.makedirs(base_out, exist_ok=True)

    print(f"\n=== Network (chunked): {network} ===")
    print(f"Input:  {base_in}")
    print(f"Output: {base_out}")

    station_dirs = sorted(
        d for d in glob.glob(os.path.join(base_in, "*"))
        if os.path.isdir(d)
    )
    n_stations_found = len(station_dirs)
    print(f"Found {n_stations_found} station directories.")

    # 1) FIRST PASS: determine dense pixels using minimal columns
    pixel_station_pairs = []

    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        for f in csv_files:
            try:
                df = pd.read_csv(f, usecols=["lat_merra2", "lon_merra2"])
            except Exception as e:
                print(f"  [WARN] Could not read minimal cols from {f}: {e}")
                continue

            pix_unique = df.dropna(subset=["lat_merra2", "lon_merra2"]).drop_duplicates()
            for _, row in pix_unique.iterrows():
                pixel_station_pairs.append(
                    (row["lat_merra2"], row["lon_merra2"], station_name)
                )

    if not pixel_station_pairs:
        print("No pixel/station info found for SNOTEL, skipping.")
        return

    df_pix_st = pd.DataFrame(
        pixel_station_pairs,
        columns=["lat_merra2", "lon_merra2", "station"]
    )

    pixel_station_counts = (
        df_pix_st.groupby(["lat_merra2", "lon_merra2"])["station"]
        .nunique()
        .reset_index(name="n_stations")
    )
    dense_pixels = pixel_station_counts.query("n_stations >= 2")[["lat_merra2", "lon_merra2"]]

    print("Total unique pixels:", len(pixel_station_counts))
    print("Dense pixels (n_stations >= 2):", len(dense_pixels))

    stations_in_dense = set()

    # Existing dense files
    if os.path.exists(dense_out):
        existing_dense_files = glob.glob(os.path.join(dense_out, f"{network}_dense_lat*_lon*.csv"))
        for f_dense in existing_dense_files:
            try:
                df_dense_existing = pd.read_csv(f_dense)
                if "stations" in df_dense_existing.columns and not df_dense_existing.empty:
                    for st in str(df_dense_existing["stations"].iloc[0]).split(","):
                        st = st.strip()
                        if st:
                            stations_in_dense.add(st)
            except Exception as e:
                print(f"  [WARN] Could not read existing dense file {f_dense}: {e}")

    n_dense_pixels_processed = 0
    n_dense_files_written = 0

    # 2) SECOND PASS: process each dense pixel separately
    if not dense_pixels.empty:
        os.makedirs(dense_out, exist_ok=True)

        for _, pix in dense_pixels.iterrows():
            plat = pix["lat_merra2"]
            plon = pix["lon_merra2"]

            out_dense = os.path.join(
                dense_out,
                f"{network}_dense_lat{plat}_lon{plon}.csv"
            )

            if os.path.exists(out_dense):
                print(f"  Skipping dense pixel lat_merra2={plat}, lon_merra2={plon} (file exists).")
                continue

            # stations that ever fall into this pixel
            stations_pix = df_pix_st[
                (df_pix_st["lat_merra2"] == plat) &
                (df_pix_st["lon_merra2"] == plon)
            ]["station"].unique()

            df_pix_all = []

            for station_name in stations_pix:
                if (network, station_name) not in pass_stations:
                    continue

                station_dir = os.path.join(base_in, station_name)
                csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
                if not csv_files:
                    continue

                for f in csv_files:
                    try:
                        df = pd.read_csv(f)
                    except Exception as e:
                        print(f"  [WARN] Could not read {f}: {e}")
                        continue

                    required_cols = {
                        "date", "time", "ts_station_k", "ts_merra2",
                        "lat_merra2", "lon_merra2",
                        "lat", "lon",
                        "cc", "lc",
                        "land_cover_group", "climate_group", "temp_class", "elev"
                    }
                    if not required_cols.issubset(df.columns):
                        continue

                    df = df[
                        (df["lat_merra2"] == plat) &
                        (df["lon_merra2"] == plon)
                    ].copy()

                    if df.empty:
                        continue

                    df["station"] = station_name
                    df_pix_all.append(df)

            if not df_pix_all:
                continue

            df_pix = pd.concat(df_pix_all, ignore_index=True)

            print(f"  Dense pixel at lat_merra2={plat}, lon_merra2={plon}, rows={len(df_pix)}")

            group_cols = ["date", "time"]
            station_list = sorted(df_pix["station"].unique())
            stations_str = ",".join(station_list)

            stations_in_dense.update(station_list)

            out_rows = []
            for (date, time), g in df_pix.groupby(group_cols):
                g_valid = g[g["ts_station_k"].notna()]
                n_valid = g_valid["station"].nunique()

                if n_valid >= 2:
                    ts_ref = g_valid["ts_station_k"].mean()
                    ts_m2  = g_valid["ts_merra2"].iloc[0]

                    lat_val  = majority(g_valid["lat"])
                    lon_val  = majority(g_valid["lon"])
                    cc_val   = majority(g_valid["cc"])
                    lc_val   = majority(g_valid["lc"])
                    lcg_val  = majority(g_valid["land_cover_group"])
                    clim_val = majority(g_valid["climate_group"])
                    tclass_val = majority(g_valid["temp_class"])

                    elev_mean = g_valid["elev"].mean() if "elev" in g_valid.columns else pd.NA
                    elevc_val = classify_elevation(elev_mean)
                else:
                    ts_ref = pd.NA
                    ts_m2  = pd.NA
                    lat_val = pd.NA
                    lon_val = pd.NA
                    cc_val = pd.NA
                    lc_val = pd.NA
                    lcg_val = pd.NA
                    clim_val = pd.NA
                    tclass_val = pd.NA
                    elev_mean = pd.NA
                    elevc_val = "DEM-Unknown"

                out_rows.append({
                    "date": date,
                    "time": time,
                    "ts_reference": ts_ref,
                    "ts_merra2": ts_m2,
                    "lat": lat_val,
                    "lon": lon_val,
                    "cc": cc_val,
                    "lc": lc_val,
                    "land_cover_group": lcg_val,
                    "climate_group": clim_val,
                    "temp_class": tclass_val,
                    "elev": elev_mean,
                    "elevation_class": elevc_val,
                })

            df_pix_mean = pd.DataFrame(out_rows)
            df_pix_mean["ts_reference"] = pd.to_numeric(df_pix_mean["ts_reference"], errors="coerce").round(3)
            df_pix_mean["ts_merra2"]    = pd.to_numeric(df_pix_mean["ts_merra2"],    errors="coerce").round(3)
            df_pix_mean["lat_merra2"] = plat
            df_pix_mean["lon_merra2"] = plon
            df_pix_mean["stations"]   = stations_str

            df_pix_mean.to_csv(out_dense, index=False)
            n_dense_pixels_processed += 1
            n_dense_files_written += 1
            print(f"    Saved dense pixel file: {out_dense}")

    else:
        print("No dense pixels found for SNOTEL.")

    # 3) SPARSE: per-station outputs for stations not in dense pixels
    print(f"Stations that appear in dense pixels: {len(stations_in_dense)}")
    print("Writing sparse (station-wise) files (excluding dense stations)...")

    n_sparse_files = 0

    for station_dir in station_dirs:
        station_name = os.path.basename(station_dir)

        if (network, station_name) not in pass_stations:
            continue
        if station_name in stations_in_dense:
            continue

        csv_files = glob.glob(os.path.join(station_dir, "*.csv"))
        if not csv_files:
            continue

        if n_sparse_files == 0:
            os.makedirs(sparse_out, exist_ok=True)

        for f in csv_files:
            out_sparse = os.path.join(sparse_out, os.path.basename(f))

            if os.path.exists(out_sparse):
                print(f"  Skipping sparse file for {network}/{station_name} ({out_sparse} exists).")
                continue

            try:
                df = pd.read_csv(f)
            except Exception as e:
                print(f"  [WARN] Could not read {f}: {e}")
                continue

            rename_map = {}
            if "ts_station_k" in df.columns:
                rename_map["ts_station_k"] = "ts_reference"
            df = df.rename(columns=rename_map)

            df.to_csv(out_sparse, index=False)
            n_sparse_files += 1

    print(f"Finished sparse files. Processed {n_sparse_files} station CSV files "
          f"(excluding {len(stations_in_dense)} dense stations).")

    print(f"All done for {network}.")
    print("===== SUMMARY =====")
    print(f"Dense pixels processed (with valid data): {n_dense_pixels_processed}")
    print(f"Dense pixel files written: {n_dense_files_written}")

# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

if __name__ == "__main__":
    pass_stations = load_pass_stations(SUMMARY_PATH)

    network_dirs = sorted(
        d for d in glob.glob(os.path.join(COMBINE_ROOT, "*"))
        if os.path.isdir(d)
    )
    networks = [os.path.basename(d) for d in network_dirs]

    print("Networks found:", networks)
    for net in networks:
        if net == "SNOTEL":
            process_network_snotel(pass_stations)
        else:
            process_network(net, pass_stations)


Networks found: ['ARM', 'BIEBRZA_S-1', 'BNZ-LTER', 'Berlin', 'COSMOS-UK', 'CTP_SMTMN', 'CW3E', 'DAHRA', 'FLUXNET-AMERIFLUX', 'FMI', 'FR_Aqui', 'HOAL', 'HOBE', 'LABFLUX', 'MAQU', 'MOL-RAO', 'NAQU', 'NGARI', 'ORACLE', 'OZNET', 'RISMA', 'ROMPS', 'Ru_CFR', 'SCAN', 'SKKU', 'SMN-SDR', 'SMOSMANIA', 'SNOTEL', 'STEMS', 'TERENO', 'TWENTE', 'USCRN', 'WEGENERNET', 'XMS-CAT']

=== Network: ARM ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/combine/ARM
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/ARM
Found 14 station directories.
Loaded 14 stations after filtering, total rows: 776154
Total unique pixels: 10
Dense pixels (n_stations >= 2): 3
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 401182


/tmp/ipykernel_90529/3872199085.py:148: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)
/tmp/ipykernel_90529/3872199085.py:148: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)
/tmp/ipykernel_90529/3872199085.py:148: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_merra2=36.5, lon_merra2=-98.125 (file exists).
  Skipping dense pixel lat_merra2=37.0, lon_merra2=-98.125 (file exists).
  Skipping dense pixel lat_merra2=37.0, lon_merra2=-97.5 (file exists).
Stations that appear in dense pixels: 7
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for ARM/Lamont-CF1 (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/ARM/sparse/Lamont-CF1_soil_temperature_depths_merra2.csv exists).
  Skipping sparse file for ARM/Marshall (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/ARM/sparse/Marshall_soil_temperature_depths_merra2.csv exists).
  Skipping sparse file for ARM/Morrison (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/ARM/sparse/Morrison_soil_temperature_depths_merra2.csv exists).
  Skipping sparse file for ARM/Newkirk (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_laye

/tmp/ipykernel_90529/3872199085.py:148: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_merra2=51.0, lon_merra2=-1.875 (file exists).
  Skipping dense pixel lat_merra2=51.5, lon_merra2=-1.25 (file exists).
  Skipping dense pixel lat_merra2=51.5, lon_merra2=0.625 (file exists).
  Skipping dense pixel lat_merra2=52.0, lon_merra2=-1.25 (file exists).
  Skipping dense pixel lat_merra2=52.0, lon_merra2=-0.625 (file exists).
  Skipping dense pixel lat_merra2=52.5, lon_merra2=0.625 (file exists).
  Skipping dense pixel lat_merra2=54.0, lon_merra2=-1.25 (file exists).
Stations that appear in dense pixels: 15
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for COSMOS-UK/AliceHolt (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/COSMOS-UK/sparse/AliceHolt_soil_temperature_depths_merra2.csv exists).
  Skipping sparse file for COSMOS-UK/Balruddery (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/COSMOS-UK/sparse/Balruddery_soil_temperature_depths

...

  Skipping dense pixel lat_merra2=67.5, lon_merra2=26.875 (file exists).
  Skipping dense pixel lat_merra2=68.5, lon_merra2=27.5 (file exists).
Stations that appear in dense pixels: 9
Writing sparse (station-wise) files (excluding dense stations)...
  Skipping sparse file for FMI/SOD140 (/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/FMI/sparse/SOD140_soil_temperature_depths_merra2.csv exists).
Finished sparse files. Processed 0 station CSV files (excluding 9 dense stations).
All done for FMI.
===== SUMMARY =====
Stations found (raw): 10
Dense pixels processed (with valid data): 0
Dense pixel files written: 0

=== Network: FR_Aqui ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/combine/FR_Aqui
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/FR_Aqui
Found 5 station directories.
Loaded 5 stations after filtering, total rows: 269718
Total unique pixels: 2
Dense pixels (n_s

/tmp/ipykernel_90529/3872199085.py:148: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_merra2=33.5, lon_merra2=101.875 (file exists).
  Skipping dense pixel lat_merra2=34.0, lon_merra2=101.875 (file exists).
  Skipping dense pixel lat_merra2=34.0, lon_merra2=102.5 (file exists).
Stations that appear in dense pixels: 16
Writing sparse (station-wise) files (excluding dense stations)...
Finished sparse files. Processed 0 station CSV files (excluding 16 dense stations).
All done for MAQU.
===== SUMMARY =====
Stations found (raw): 16
Dense pixels processed (with valid data): 0
Dense pixel files written: 0

=== Network: MOL-RAO ===
Input:  /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/combine/MOL-RAO
Output: /home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean/MOL-RAO
Found 2 station directories.
Loaded 2 stations after filtering, total rows: 115752
Total unique pixels: 2
Dense pixels (n_stations >= 2): 0
Processing dense pixels (pixel-mean files)...
Rows in dense pixels (before grouping): 0
N

/tmp/ipykernel_90529/3872199085.py:148: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dense_existing = pd.read_csv(f_dense)


  Skipping dense pixel lat_merra2=18.0, lon_merra2=-66.875 (file exists).
  Skipping dense pixel lat_merra2=20.0, lon_merra2=-155.625 (file exists).
  Skipping dense pixel lat_merra2=32.5, lon_merra2=-85.625 (file exists).
  Skipping dense pixel lat_merra2=33.0, lon_merra2=-91.25 (file exists).
  Skipping dense pixel lat_merra2=33.0, lon_merra2=-90.625 (file exists).
  Skipping dense pixel lat_merra2=33.5, lon_merra2=-102.5 (file exists).
  Skipping dense pixel lat_merra2=33.5, lon_merra2=-90.625 (file exists).
  Skipping dense pixel lat_merra2=34.0, lon_merra2=-90.625 (file exists).
  Skipping dense pixel lat_merra2=34.0, lon_merra2=-90.0 (file exists).
  Skipping dense pixel lat_merra2=35.0, lon_merra2=-119.375 (file exists).
  Skipping dense pixel lat_merra2=35.0, lon_merra2=-86.875 (file exists).
  Skipping dense pixel lat_merra2=35.0, lon_merra2=-86.25 (file exists).
  Skipping dense pixel lat_merra2=36.5, lon_merra2=-115.625 (file exists).
  Skipping dense pixel lat_merra2=37.5, 

## Pixel_mean files validation

### 1. dense/sparse files checking

In [2]:
import pandas as pd
from pathlib import Path
import glob
import os

COMBINE_ROOT = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/combine")
OUT_ROOT     = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/pixel_mean")
SUMMARY_PATH = Path("/home/aminr/Desktop/Master_Thesis/Global_scale/sub_surface_layers/merra2/merra2_insitu_sd_corr_summary.csv")

summary = pd.read_csv(SUMMARY_PATH)

# Stations that passed both filters
passed = summary[(summary["pass_sd_filter"] == True) & (summary["pass_corr_filter"] == True)]
passed_set = set(zip(passed["network"], passed["station"]))

# Stations that failed any filter
failed = summary[(summary["pass_sd_filter"] == False) | (summary["pass_corr_filter"] == False)]
failed_set = set(zip(failed["network"], failed["station"]))

dense_stations = set()
sparse_stations = set()

for net_dir in sorted(COMBINE_ROOT.iterdir()):
    if not net_dir.is_dir():
        continue
    network = net_dir.name
    net_out = OUT_ROOT / network
    dense_dir = net_out / "dense"
    sparse_dir = net_out / "sparse"

    # collect from dense
    if dense_dir.exists():
        for f in glob.glob(str(dense_dir / f"{network}_dense_lat*_lon*.csv")):
            df = pd.read_csv(f)
            if "stations" in df.columns and not df.empty:
                for st in str(df["stations"].iloc[0]).split(","):
                    st = st.strip()
                    if st:
                        dense_stations.add((network, st))

    # collect from sparse
    if sparse_dir.exists():
        for st_dir in sorted((COMBINE_ROOT / network).iterdir()):
            if not st_dir.is_dir():
                continue
            station = st_dir.name
            pattern = str(sparse_dir / f"{station}*.csv")
            files = glob.glob(pattern)
            if files:
                sparse_stations.add((network, station))

print("Total passed stations        :", len(passed_set))
print("Stations in dense pixels     :", len(dense_stations))
print("Stations in sparse outputs   :", len(sparse_stations))

print("\nStations that passed but not in dense or sparse:")
missing = passed_set - dense_stations - sparse_stations
print(missing)

print("\nStations that failed but appear in dense or sparse (should be empty):")
bad = (failed_set & dense_stations) | (failed_set & sparse_stations)
print(bad)


/tmp/ipykernel_90529/2362303383.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_90529/2362303383.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_90529/2362303383.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_90529/2362303383.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_90529/2362303383.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)
/tmp/ipykernel_90529/2362303383.py:34: DtypeWarning: Columns (6,8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read

Total passed stations        : 1156
Stations in dense pixels     : 700
Stations in sparse outputs   : 457

Stations that passed but not in dense or sparse:
set()

Stations that failed but appear in dense or sparse (should be empty):
set()
